<a href="https://colab.research.google.com/github/riya-shrn/AI-ML-PROJECTS/blob/level/CEN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

importing dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from sklearn.metrics import accuracy_score

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=64, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 42.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.64MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.34MB/s]


In [ ]:
class FeatureCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3),   # 26x26
            nn.ReLU(),
            nn.MaxPool2d(2),      # 13x13
            nn.Conv2d(16, 32, 3), # 11x11
            nn.ReLU(),
            nn.MaxPool2d(2)       # 5x5
        )

    def forward(self, x):
        x = self.net(x)
        return x.view(x.size(0), -1)



In [ ]:
feature_model = FeatureCNN()
temp_fc = nn.Linear(32 * 5 * 5, 10)

model = nn.Sequential(feature_model, temp_fc)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):  # VERY FEW epochs
    for images, labels in train_loader:
        optimizer.zero_grad()
        out = model(images)
        loss = loss_fn(out, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} done")



Epoch 1 done
Epoch 2 done
Epoch 3 done


In [ ]:
feature_model.eval()

def get_features(loader):
    X, y = [], []
    with torch.no_grad():
        for images, labels in loader:
            features = feature_model(images)
            X.append(features.numpy())
            y.append(labels.numpy())
    return np.vstack(X), np.hstack(y)

X_train, y_train = get_features(train_loader)
X_test, y_test   = get_features(test_loader)


In [ ]:
models = []
for _ in range(3):  # VERY SMALL ensemble
    X_s, y_s = resample(X_train, y_train)
    clf = RandomForestClassifier(n_estimators=50)
    clf.fit(X_s, y_s)
    models.append(clf)


In [ ]:
preds = np.array([m.predict(X_test) for m in models])

final_pred = np.apply_along_axis(
    lambda x: np.bincount(x).argmax(), axis=0, arr=preds
)

acc = accuracy_score(y_test, final_pred)
print("CEN Accuracy:", acc)


CEN Accuracy: 0.9843
